# drfn: the Ge'ez ByT5 root analyser as the Ge'ez-side tokenizer

Same data, trainer (version D, advisor-update-2019), settings and seeds as the 2019 BPE runs; the Amharic side stays BPE.

**Pipeline**
1. The analyser (weights from the attached output of `geez-morphology-byt5-training`) reads every distinct Ge'ez word of
   train + dev once. Readings go to `runs/shared/analyser_cache.json`; root + pattern labels to `analyser_labels.json`.
2. Pre-flight check (no GPU training yet): how many words enter as readings, source lengths against BPE, how many
   sentences pass the 128-token limit. The notebook stops here if more than 1% would be cut.
3. Runs. Ge'ez input: a word whose root occurs at least twice in train enters as two tokens, `[root:…]` and one
   morphology token `[m:stem=…|tense=…|person=…|gender=…|number=…]`; every other word keeps its BPE pieces.
   * `input_plain_s11`, `_s22`, `_s33`: plain model. Compare with Plain BPE.
   * `input_labels_s11`: the same input plus root + pattern heads on the analyser labels (near trivial, a check).

   The transformer body is fixed to the 2019 widths (FFN 1186 without heads, 1169 with heads); the only extra parameters
   are the embedding rows of the root and morphology tokens. The trainer's parameter matching would otherwise shrink the FFN.

Inputs to attach: dataset `drfn-analyser-pack` (latest version), dataset `geez-morphology-training-inputs`, output of
notebook `geez-morphology-byt5-training`. GPU on, Internet on.

In [ ]:
SEEDS = "11 22 33"   # input_plain runs, seed 11 first
LABELS_RUN = True     # input_labels_s11 after the plain runs
CACHE_ONLY = False    # True: stop after the pre-flight check


In [ ]:
!pip install -q transformers==5.19.0 accelerate==1.15.0 tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
import glob, os, shutil, subprocess, sys, json
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat + ' (attach the input that holds it)')
    return hits[0]
pack_src = os.path.dirname(find('run_analyser_input.sh'))
bundle   = find('geez-morphology_3dc784e.bundle')
weights  = os.path.dirname(find('best_weights_by_val_score/model.safetensors'))
print('pack   ', pack_src); print('bundle ', bundle); print('weights', weights)
fin = os.path.join(os.path.dirname(weights), 'RUN_FINISHED.txt')
print(open(fin).read() if os.path.exists(fin) else '(no RUN_FINISHED.txt)')
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
if not os.path.exists('/tmp/geez_morph'):
    subprocess.run(['git', 'clone', '-q', bundle, '/tmp/geez_morph'], check=True)
    subprocess.run(['git', '-C', '/tmp/geez_morph', 'checkout', '-q', '3dc784eb7c41133712560c33ebdbd7f80d2f0634'], check=True)
print(subprocess.run(['git', '-C', '/tmp/geez_morph', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
print(subprocess.run(['sha256sum', 'train.py'], capture_output=True, text=True).stdout)

In [ ]:
# Step 1: the analyser reads every distinct Ge'ez word once; readings + label cache go to runs/shared/
def stream(cmd, env=None, skip_steps=False):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if skip_steps and (line.startswith('{"step"') or line.startswith("{'step'")): continue
        print(line, end='')
    return p.wait()
if not os.path.exists('runs/shared/analyser_labels.json'):
    if stream([sys.executable, '-u', 'build_analyser_cache.py', '--data-dir', 'data', '--morphology-repo', '/tmp/geez_morph',
               '--model-dir', weights, '--out-dir', 'runs/shared']):
        raise RuntimeError('cache build failed')
else:
    print('label cache exists, kept')

In [ ]:
# What the analyser gave
c = json.load(open('runs/shared/analyser_cache.json', encoding='utf-8')); L = json.load(open('runs/shared/analyser_labels.json', encoding='utf-8'))
print(c['meta']); print(L['metadata'])
print('patterns:', L['pattern_classes'][1:])
print('top roots:', L['root_classes'][1:21])
r = c['readings']
for w in list(r)[::max(1, len(r)//15)][:15]: print(w, '->', r[w])


In [ ]:
# Step 2: pre-flight check, before any GPU training
sys.path.insert(0, PACK)
from data import load_pairs, train_bpe
from tokenizers import Tokenizer
from analyser_tokenizer import AnalyserInputTokenizer
from collections import Counter
import statistics as st
tr, _ = load_pairs('data/age_train.tsv'); dv, _ = load_pairs('data/age_dev.tsv')
if not os.path.exists('runs/shared/preflight_bpe.json'):
    train_bpe(tr, 16000, 'runs/shared/preflight_bpe.json')
bpe = Tokenizer.from_file('runs/shared/preflight_bpe.json')
readings = json.load(open('runs/shared/analyser_cache.json', encoding='utf-8'))['readings']
train_words = [w for s, _ in tr for w in s.split()]
tk = AnalyserInputTokenizer(bpe, readings, train_words, 16000, 2)
print(f'train tokens entering as readings: {tk.n_analysed}/{tk.n_words} ({tk.n_analysed/tk.n_words:.1%})')
print(f'source ids: 16000 BPE + {tk.n_root_tokens} root tokens + {tk.n_morph_tokens} morphology tokens = {len(tk)}')
def lens(pairs, enc): return [2 + sum(len(enc(w)) for w in s.split()) for s, _ in pairs]
for name, pairs in (('train', tr), ('dev', dv)):
    lb = lens(pairs, lambda w: bpe.encode(w).ids); la = lens(pairs, lambda w: tk.encode(w).ids)
    print(f'{name}: mean length BPE {st.mean(lb):.1f} -> analyser input {st.mean(la):.1f}; '
          f'over 128: BPE {sum(l > 128 for l in lb)}, analyser input {sum(l > 128 for l in la)} of {len(pairs)}')
    cut = sum(l > 128 for l in la) / len(pairs)
    if cut > 0.01: raise RuntimeError(f'{cut:.1%} of {name} sentences would be cut at 128 tokens; not training')
dev_unk = sum(1 for s, _ in dv for w in s.split() if 3 in tk.encode(w).ids)
print('dev words with an unseen morphology token ([UNK]):', dev_unk)
print('\nmost frequent train words entering as readings (check names here):')
for w, n in Counter(w for w in train_words if tk._analysed(w)).most_common(30): print(f'{n:6d}  {w}  ->  {" ".join(tk.tokens(w))}')
print('\nmost frequent train words kept as BPE pieces:')
for w, n in Counter(w for w in train_words if not tk._analysed(w)).most_common(15): print(f'{n:6d}  {w}  ->  {" ".join(tk.tokens(w))}')
if CACHE_ONLY: raise RuntimeError('CACHE_ONLY: stopped after the pre-flight check')

In [ ]:
# Step 3: the runs (each writes train_log.jsonl, metadata.json, best.pt, last.pt in runs/<name>/)
env = dict(os.environ, SEEDS=SEEDS, LABELS_RUN='1' if LABELS_RUN else '0')
print('exit', stream(['bash', 'run_analyser_input.sh'], env=env, skip_steps=True))

In [ ]:
# Summary next to the advisor-update-2019 numbers (best dev loss; greedy chrF++ on the first 256 dev sentences at that epoch)
base = {'Plain BPE (FFN 1186, 20,341,648 params)': {11: (4.4410, 9.32), 22: (4.4267, 10.24), 33: (4.4257, 9.92)},
        'Both labels geezRPA (FFN 1169)': {11: (4.3537, 11.44), 22: (4.3459, 11.41), 33: (4.3566, 11.35)}}
rows = []
for d in sorted(glob.glob('runs/input_*')):
    ev = [json.loads(l) for l in open(os.path.join(d, 'train_log.jsonl')) if '"dev_loss"' in l]
    if not ev: continue
    m = json.load(open(os.path.join(d, 'metadata.json')))
    b = min(ev, key=lambda e: e['dev_loss'])
    rows.append((os.path.basename(d), len(ev), round(b['dev_loss'], 4), round(b['dev_chrf'], 2), b['epoch'],
                 m['cfg']['ffn_dim'], m['params'], m['cfg'].get('src_vocab_size')))
print('%-18s %6s %9s %7s %5s %5s %11s %9s' % ('run', 'epochs', 'dev_loss', 'chrF++', 'best', 'ffn', 'params', 'src_vocab'))
for r_ in rows: print('%-18s %6d %9.4f %7.2f %5d %5d %11d %9d' % r_)
print(); [print(k, v) for k, v in base.items()]
open('/kaggle/working/summary.txt', 'w').write('\n'.join('\t'.join(map(str, r_)) for r_ in rows) + '\n')
